### Bibliotecas

In [1]:
import pandas as pd
import numpy as np

import warnings
import os

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Definindo diretório
os.chdir("C:\\Users\\Usuaro\\OneDrive\\Área de Trabalho\\play_ground")

# Exibir todas as linhas e colunas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Ignorar avisos
warnings.filterwarnings("ignore")

### Tratamento dos dados

In [3]:
# Importando os dados
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv('sample_submission.csv')

In [4]:
from sklearn.model_selection import train_test_split

target = "Listening_Time_minutes"
X = train.drop(columns=[target])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=0, shuffle=True)

### Dados faltantes

In [5]:
# Observaçções faltantes por colunas
md = pd.concat([
    X_train.isna().sum().sort_values(ascending=False),
    X_valid.isna().sum().sort_values(ascending=False)
], axis=1, keys=['X_train', 'X_valid'])

print(md)

                             X_train  X_valid
Guest_Popularity_percentage   116866    29164
Episode_Length_minutes         69676    17417
Number_of_Ads                      1        0
id                                 0        0
Podcast_Name                       0        0
Episode_Title                      0        0
Genre                              0        0
Host_Popularity_percentage         0        0
Publication_Day                    0        0
Publication_Time                   0        0
Episode_Sentiment                  0        0


### Valores Faltantes

##### Episode_Length_minutes e Guest_Popularity_percentage

In [6]:
def substituir_valores_faltantes_por_mediana(df_train, df_valid, col, grupo='Podcast_Name'):
    # Calcular a mediana por podcast no conjunto de treino (X_train)
    medianas_por_podcast = df_train.groupby(grupo)[col].median()

    # Substituir valores faltantes no conjunto de treino (X_train) usando a mediana do grupo
    for podcast, mediana in medianas_por_podcast.items():
        # Substituir valores faltantes no conjunto de treino
        df_train.loc[(df_train[grupo] == podcast) & (df_train[col].isna()), col] = mediana

    # Substituir valores faltantes no conjunto de validação (X_valid) usando a mediana do grupo calculada em X_train
    for podcast, mediana in medianas_por_podcast.items():
        # Substituir valores faltantes no conjunto de validação
        df_valid.loc[(df_valid[grupo] == podcast) & (df_valid[col].isna()), col] = mediana

    return df_train, df_valid

# Agora você pode chamar a função para qualquer coluna
X_train, X_valid = substituir_valores_faltantes_por_mediana(X_train, X_valid, col='Episode_Length_minutes')
X_train, X_valid = substituir_valores_faltantes_por_mediana(X_train, X_valid, col='Guest_Popularity_percentage')

##### Number_of_Ads

In [7]:
def substituir_valores_faltantes_por_moda(df_train, df_valid, col, grupo='Podcast_Name'):
    # Calcular a moda por podcast no conjunto de treino
    modas_por_podcast = df_train.groupby(grupo)[col].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

    # Substituir valores faltantes no conjunto de treino
    for podcast, moda in modas_por_podcast.items():
        df_train.loc[(df_train[grupo] == podcast) & (df_train[col].isna()), col] = moda

    # Substituir valores faltantes no conjunto de validação
    for podcast, moda in modas_por_podcast.items():
        df_valid.loc[(df_valid[grupo] == podcast) & (df_valid[col].isna()), col] = moda

    return df_train, df_valid

X_train, X_valid = substituir_valores_faltantes_por_moda(X_train, X_valid, col='Number_of_Ads')

### Outliers

##### Episode_Length_minutes

In [8]:
def substituir_outliers_por_mediana(df_train, df_valid, col='Episode_Length_minutes', grupo='Podcast_Name', limite=150):
    # Calcular as medianas por grupo no conjunto de treino, excluindo outliers
    medianas_por_podcast = df_train[df_train[col] <= limite].groupby(grupo)[col].median()

    # Substitui os outliers no conjunto de treino
    outlier_idx_train = df_train[df_train[col] > limite].index
    for idx in outlier_idx_train:
        podcast = df_train.loc[idx, grupo]
        # Recupera a mediana do podcast, se houver
        if podcast in medianas_por_podcast:
            df_train.at[idx, col] = medianas_por_podcast[podcast]

    # Substitui os outliers no conjunto de validação, usando as medianas calculadas no conjunto de treino
    outlier_idx_valid = df_valid[df_valid[col] > limite].index
    for idx in outlier_idx_valid:
        podcast = df_valid.loc[idx, grupo]
        # Recupera a mediana do podcast, se houver
        if podcast in medianas_por_podcast:
            df_valid.at[idx, col] = medianas_por_podcast[podcast]

    return df_train, df_valid

# Aplicando a função no conjunto de dados
X_train, X_valid = substituir_outliers_por_mediana(X_train, X_valid)

##### Host_Popularity_percentage e Guest_Popularity_percentage

In [9]:
# Função para aplicar limite superior
def limitar_percentuais(df, colunas, limite_superior=100):
    for col in colunas:
        df[col] = df[col].clip(upper=limite_superior)
    return df

colunas_limite = ['Host_Popularity_percentage', 'Guest_Popularity_percentage']

X_train = limitar_percentuais(X_train, colunas_limite)
X_valid = limitar_percentuais(X_valid, colunas_limite)

##### Number_of_Ads

In [10]:
def substituir_outliers_por_moda(df_train, df_valid, col='Number_of_Ads', grupo='Podcast_Name', limite_superior=5):
    # Criar uma máscara de valores considerados "não-outliers"
    nao_outliers = df_train[col] <= limite_superior

    # Calcular a moda por grupo (podcast) apenas com os dados não-outliers do treino
    modas_por_podcast = (
        df_train[nao_outliers]
        .groupby(grupo)[col]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )

    # Substituir outliers no treino
    for podcast, moda in modas_por_podcast.items():
        mask = (df_train[grupo] == podcast) & (df_train[col] > limite_superior)
        df_train.loc[mask, col] = moda

    # Substituir outliers na validação usando as modas do treino
    for podcast, moda in modas_por_podcast.items():
        mask = (df_valid[grupo] == podcast) & (df_valid[col] > limite_superior)
        df_valid.loc[mask, col] = moda

    return df_train, df_valid


X_train, X_valid = substituir_outliers_por_moda(
    X_train, X_valid,
    col='Number_of_Ads',
    grupo='Podcast_Name',
    limite_superior=5
)

### Criação de variáveis

In [11]:
# Removendo "episode" e espaços extras
X_train['Episode_Title'] = X_train['Episode_Title'].str.replace(r'(?i)episode\s*', '', regex=True).str.strip()
X_valid['Episode_Title'] = X_valid['Episode_Title'].str.replace(r'(?i)episode\s*', '', regex=True).str.strip()

# Convertendo para numérico (ignora erros se houver algum texto estranho)
X_train['Episode_Title'] = pd.to_numeric(X_train['Episode_Title'], errors='coerce')
X_valid['Episode_Title'] = pd.to_numeric(X_valid['Episode_Title'], errors='coerce')

In [12]:
# Criando uma nova coluna combinando Number_of_Ads e Episode_Sentiment
X_train['Ads_Sentiment_Combo'] = X_train['Number_of_Ads'].astype(str) + '_' + X_train['Episode_Sentiment']
X_valid['Ads_Sentiment_Combo'] = X_valid['Number_of_Ads'].astype(str) + '_' + X_valid['Episode_Sentiment']

In [13]:
# Host e Guest

# Bins e rótulos
bins = [0, 25, 50, 75, 100]
labels = ['0_25', '26_50', '51_75', '76_100']

# Criando as categorias com pd.cut
X_train['Host_Popularity_category'] = pd.cut(X_train['Host_Popularity_percentage'], bins=bins, labels=labels, include_lowest=True)
X_train['Guest_Popularity_category'] = pd.cut(X_train['Guest_Popularity_percentage'], bins=bins, labels=labels, include_lowest=True)

X_valid['Host_Popularity_category'] = pd.cut(X_valid['Host_Popularity_percentage'], bins=bins, labels=labels, include_lowest=True)
X_valid['Guest_Popularity_category'] = pd.cut(X_valid['Guest_Popularity_percentage'], bins=bins, labels=labels, include_lowest=True)

In [14]:
# Aplicando get_dummies nas novas categorias
X_train_dummies = pd.get_dummies(X_train, columns=['Genre', 'Publication_Time', 'Publication_Day', 'Ads_Sentiment_Combo',
                                                  'Host_Popularity_category', 'Guest_Popularity_category'], drop_first=True)

X_valid_dummies = pd.get_dummies(X_valid, columns=['Genre', 'Publication_Time', 'Publication_Day', 'Ads_Sentiment_Combo',
                                                  'Host_Popularity_category', 'Guest_Popularity_category'], drop_first=True)

# Reindexando X_valid para ter as mesmas colunas que X_train
X_valid_dummies = X_valid_dummies.reindex(columns=X_train_dummies.columns, fill_value=0)

# Convertendo booleanos para inteiros (caso necessário)
for column in X_train_dummies.columns:
    if X_train_dummies[column].dtype == 'bool':
        X_train_dummies[column] = X_train_dummies[column].astype(int)
        X_valid_dummies[column] = X_valid_dummies[column].astype(int)

In [15]:
# Dropar as colunas 'id', 'Podcast_Name', 'Episode_Title', 'Publication_Day' dos DataFrames
X_train_dummies = X_train_dummies.drop(columns=['id', 'Podcast_Name', 'Episode_Sentiment', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])

X_valid_dummies = X_valid_dummies.drop(columns=['id', 'Podcast_Name', 'Episode_Sentiment', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])

### Padronizando os dados

In [16]:
from sklearn.preprocessing import MinMaxScaler

# Inicializando o escalonador Min-Max
scaler = MinMaxScaler()

# Escalonando os dados de treino e validação
X_train_scaled = scaler.fit_transform(X_train_dummies)
X_valid_scaled = scaler.transform(X_valid_dummies)

# Convertendo os resultados de volta para DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_dummies.columns)
X_valid_scaled = pd.DataFrame(X_valid_scaled, columns=X_valid_dummies.columns)

### Previsão

In [17]:
"""from sklearn.ensemble import RandomForestRegressor

# Modelo base
rf_model = RandomForestRegressor(random_state=42)

# Grade de parâmetros ajustada
param_grid = {
    'n_estimators': [100, 200],       
    'max_depth': [10, 20, None],            
    'min_samples_split': [2, 4, 5]       
}

# Busca em grade com validação cruzada
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,  # Pode usar 3 em vez de 5 para acelerar
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=3
)

# Treinamento
grid_search.fit(X_train_scaled, y_train)

# Melhor modelo
best_rf_model = grid_search.best_estimator_
y_pred = best_rf_model.predict(X_valid_scaled)

# Avaliação
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))

# Resultados
print("Melhores parâmetros:", grid_search.best_params_)
print(f"RMSE: {rmse:.4f}")"""

'from sklearn.ensemble import RandomForestRegressor\n\n# Modelo base\nrf_model = RandomForestRegressor(random_state=42)\n\n# Grade de parâmetros ajustada\nparam_grid = {\n    \'n_estimators\': [100, 200],       # Mais árvores pode melhorar a performance sem aumentar muito o custo\n    \'max_depth\': [10, 20, None],            # Aumenta a capacidade de modelagem, sem exagerar\n    \'min_samples_split\': [2, 4, 5]       # Reduz overfitting sem limitar muito as divisões\n}\n\n# Busca em grade com validação cruzada\ngrid_search = GridSearchCV(\n    estimator=rf_model,\n    param_grid=param_grid,\n    cv=5,  # Pode usar 3 em vez de 5 para acelerar\n    scoring=\'neg_root_mean_squared_error\',\n    n_jobs=-1,\n    verbose=3\n)\n\n# Treinamento\ngrid_search.fit(X_train_scaled, y_train)\n\n# Melhor modelo\nbest_rf_model = grid_search.best_estimator_\ny_pred = best_rf_model.predict(X_valid_scaled)\n\n# Avaliação\nrmse = np.sqrt(mean_squared_error(y_valid, y_pred))\n\n# Resultados\nprint("Melhor

In [47]:
"""# Exemplo de como utilizar o melhor parâmetro achados acima
best_params = {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}

# Criando e treinando o modelo com os melhores parâmetros
rf_model = RandomForestRegressor(**best_params, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Previsão e avaliação
y_pred = rf_model.predict(X_valid_scaled)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))

print(f"RMSE com melhores parâmetros reutilizados: {rmse:.4f}")"""

'# Exemplo de como utilizar o melhor parâmetro achados acima\nbest_params = {\'max_depth\': None, \'min_samples_split\': 5, \'n_estimators\': 200}\n\n# Criando e treinando o modelo com os melhores parâmetros\nrf_model = RandomForestRegressor(**best_params, random_state=42)\nrf_model.fit(X_train_scaled, y_train)\n\n# Previsão e avaliação\ny_pred = rf_model.predict(X_valid_scaled)\nrmse = np.sqrt(mean_squared_error(y_valid, y_pred))\n\nprint(f"RMSE com melhores parâmetros reutilizados: {rmse:.4f}")'

In [17]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge

# Modelos base com parâmetros ajustados para RandomForest
estimators = [
    ('rf', RandomForestRegressor(
        max_depth=None,
        min_samples_split=5,
        n_estimators=200,
        random_state=42
    )),
    ('gb', GradientBoostingRegressor(random_state=42))  # parâmetros padrão
]

# Meta-modelo
final_estimator = Ridge()

# Modelo empilhado (stacking)
stacked_model = StackingRegressor(
    estimators=estimators,
    final_estimator=final_estimator,
    cv=5,
    verbose=3
)

# Treinamento
stacked_model.fit(X_train_scaled, y_train)

# Previsão e avaliação
y_pred_stack = stacked_model.predict(X_valid_scaled)
rmse_stack = np.sqrt(mean_squared_error(y_valid, y_pred_stack))

# Resultado
print(f"RMSE com modelo empilhado (RandomForest ajustado): {rmse_stack:.4f}")

RMSE com modelo empilhado (RandomForest ajustado): 12.8738


### Teste

In [21]:
# Copiar e prepara X_test
X_test = test.copy()
test_ids = X_test["id"]

# Substituir valores ausentes por mediana por grupo
X_train, X_test = substituir_valores_faltantes_por_mediana(X_train, X_test, col='Episode_Length_minutes')
X_train, X_test = substituir_valores_faltantes_por_mediana(X_train, X_test, col='Guest_Popularity_percentage')

# Substituir valores ausentes por moda por grupo para Number_of_Ads
X_train, X_test = substituir_valores_faltantes_por_moda(X_train, X_test, col='Number_of_Ads')

# Substituir outliers por mediana e limitar percentuais
X_train, X_test = substituir_outliers_por_mediana(X_train, X_test)
X_test = limitar_percentuais(X_test, colunas_limite)

# Substituir outliers em Number_of_Ads por moda
X_train, X_test = substituir_outliers_por_moda(X_train, X_test, col='Number_of_Ads', grupo='Podcast_Name', limite_superior=5)

# Tratar Episode_Title
X_test['Episode_Title'] = X_test['Episode_Title'].str.replace(r'(?i)episode\s*', '', regex=True).str.strip()
X_test['Episode_Title'] = pd.to_numeric(X_test['Episode_Title'], errors='coerce')

# Criar coluna combinada e categorias por faixa
X_test['Ads_Sentiment_Combo'] = X_test['Number_of_Ads'].astype(str) + '_' + X_test['Episode_Sentiment']

# Bins e rótulos
bins = [0, 25, 50, 75, 100]
labels = ['0_25', '26_50', '51_75', '76_100']

X_test['Host_Popularity_category'] = pd.cut(X_test['Host_Popularity_percentage'], bins=bins, labels=labels, include_lowest=True)
X_test['Guest_Popularity_category'] = pd.cut(X_test['Guest_Popularity_percentage'], bins=bins, labels=labels, include_lowest=True)

# Gerar dummies (mesmas colunas de treino)
X_test_dummies = pd.get_dummies(X_test, columns=['Genre', 'Publication_Time', 'Publication_Day', 'Ads_Sentiment_Combo',
                                                  'Host_Popularity_category', 'Guest_Popularity_category'], drop_first=True)

# Reindexar para ter as mesmas colunas do treino
X_test_dummies = X_test_dummies.reindex(columns=X_train_dummies.columns, fill_value=0)

# Garantir booleanos como int
for column in X_test_dummies.columns:
    if X_test_dummies[column].dtype == 'bool':
        X_test_dummies[column] = X_test_dummies[column].astype(int)

#X_test_dummies = X_test_dummies.drop(columns=['id', 'Podcast_Name', 'Episode_Sentiment', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])

# Escalonar X_test com o mesmo scaler treinado em X_train
X_test_scaled = scaler.transform(X_test_dummies)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_dummies.columns)

# previsão
y_test_pred = stacked_model.predict(X_test_scaled)

### Submissão

In [30]:
# Submissão
submission = pd.read_csv("sample_submission.csv")
submission["Listening_Time_minutes"] = y_test_pred
submission["id"] = test_ids  # garante que os ids estão corretos
submission.to_csv("stacked_model_final.csv", index=False)